In [145]:
from config import *
import pandas as pd
import torchaudio
import random
import re
from pathlib import Path
OUTPUT_DIR.mkdir(exist_ok=True)

In [146]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Load Metadata**

Load the original metadata files for dementia and non-dementia speakers, which contain speaker information and dataset split hints used to guide dataset construction.

In [147]:
dementia_meta = pd.read_csv(DEMENTIA_META)
nodementia_meta = pd.read_csv(NODEMENTIA_META)

print("Dementia metadata shape:", dementia_meta.shape)
print("No dementia metadata shape:", nodementia_meta.shape)

dementia_meta.head()

Dementia metadata shape: (84, 16)
No dementia metadata shape: (61, 10)


,name,dementia type,birth,death,first symptoms,URLs after symptoms,5 years,5 < 10 years,10 < 15 years,gender,ethnicity,datasplit,language,unknown 1,unkown 2,unknown 3
0,Abe Burrows,Alzheimer,1910,1985,1975.0,NaN,https://www.youtube.com/watch?v=VezbsmCriw4,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
1,Aileen Hernandez,Dementia,1926,2017,2012.0,https://youtu.be/x7hujcEhQuY,https://youtu.be/CshhDl-YwkY \nhttps://youtu.b...,NaN,NaN,female,Black/African American,train,NaN,NaN,NaN,NaN
2,Alan Ramsey,Dementia,1938,2020,2015.0,NaN,https://www.youtube.com/watch?v=CHeXE4c6EDI,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
3,Allan Burns,Lewy body,1935,2021,NaN,NaN,https://www.youtube.com/watch?v=aD3hL-kWoPc,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
4,Andrew Sachs,Dementia,1930,2016,2012.0,NaN,NaN,https://youtu.be/FSgKLooW1LM,https://youtu.be/3V1iFmavqG4,male,NaN,train,NaN,NaN,NaN,NaN


**Normalize Speaker Names**

Standardize speaker names by converting them to a consistent format so that names in the metadata match the corresponding audio folder names.

In [160]:
import re

def normalize(name):
    name = str(name).lower()
    name = re.sub(r"[^\w]", "", name)
    return name

dementia_meta["speaker"] = dementia_meta["name"].apply(normalize)
nodementia_meta["speaker"] = nodementia_meta["name"].apply(normalize)

dementia_meta["label"] = "dementia"
nodementia_meta["label"] = "nodementia"

**Combine Metadata**

Merge the dementia and non-dementia metadata tables into a single dataframe so all speakers can be processed together.

In [149]:
metadata = pd.concat([dementia_meta, nodementia_meta], ignore_index=True)

print(metadata.columns)
metadata.head()

Index(['name', 'dementia type', 'birth', 'death', 'first symptoms',
       'URLs after symptoms', '5 years', '5 < 10 years', '10 < 15 years',
       'gender', 'ethnicity', 'datasplit', 'language', 'unknown 1', 'unkown 2',
       'unknown 3', 'speaker', 'label', 'birthdate', 'deathdate',
       'age above 70', 'age above 55 less than 70', 'age less than 55'],
      dtype='object')


,name,dementia type,birth,death,first symptoms,URLs after symptoms,5 years,5 < 10 years,10 < 15 years,gender,...,unknown 1,unkown 2,unknown 3,speaker,label,birthdate,deathdate,age above 70,age above 55 less than 70,age less than 55
0,Abe Burrows,Alzheimer,1910.0,1985,1975.0,NaN,https://www.youtube.com/watch?v=VezbsmCriw4,NaN,NaN,male,...,NaN,NaN,NaN,abeburrows,dementia,NaN,NaN,NaN,NaN,NaN
1,Aileen Hernandez,Dementia,1926.0,2017,2012.0,https://youtu.be/x7hujcEhQuY,https://youtu.be/CshhDl-YwkY \nhttps://youtu.b...,NaN,NaN,female,...,NaN,NaN,NaN,aileenhernandez,dementia,NaN,NaN,NaN,NaN,NaN
2,Alan Ramsey,Dementia,1938.0,2020,2015.0,NaN,https://www.youtube.com/watch?v=CHeXE4c6EDI,NaN,NaN,male,...,NaN,NaN,NaN,alanramsey,dementia,NaN,NaN,NaN,NaN,NaN
3,Allan Burns,Lewy body,1935.0,2021,NaN,NaN,https://www.youtube.com/watch?v=aD3hL-kWoPc,NaN,NaN,male,...,NaN,NaN,NaN,allanburns,dementia,NaN,NaN,NaN,NaN,NaN
4,Andrew Sachs,Dementia,1930.0,2016,2012.0,NaN,NaN,https://youtu.be/FSgKLooW1LM,https://youtu.be/3V1iFmavqG4,male,...,NaN,NaN,NaN,andrewsachs,dementia,NaN,NaN,NaN,NaN,NaN


**Build Speaker Split Lists**

Create lists of speakers assigned to the train, validation, and test splits based on the metadata datasplit column.

In [150]:
train_speakers = set(metadata[metadata["datasplit"] == "train"]["speaker"])
valid_speakers = set(metadata[metadata["datasplit"] == "valid"]["speaker"])
test_speakers = set(metadata[metadata["datasplit"] == "test"]["speaker"])

print("train speakers:", len(train_speakers))
print("valid speakers:", len(valid_speakers))
print("test speakers:", len(test_speakers))

train speakers: 118
valid speakers: 25
test speakers: 2


**Fix Broken Test Split**

Adjust the dataset splits by moving unusable test speakers and sampling a new balanced test set to ensure both classes are represented.

In [151]:
valid_speakers = valid_speakers.union(test_speakers)
test_speakers = set()

train_meta = metadata[metadata["speaker"].isin(train_speakers)]

dementia_train = train_meta[train_meta["label"] == "dementia"]["speaker"].unique()
nodementia_train = train_meta[train_meta["label"] == "nodementia"]["speaker"].unique()

n_test = int(min(len(dementia_train), len(nodementia_train)) * 0.15)

test_dementia = random.sample(list(dementia_train), n_test)
test_nodementia = random.sample(list(nodementia_train), n_test)

test_speakers = set(test_dementia) | set(test_nodementia)

train_speakers = train_speakers - test_speakers

**Scan Audio Files**

Search the audio directories for .wav files, validate that they can be loaded, and collect information about each recording.

In [152]:
rows = []
invalid_files = []

def process_folder(folder, label):

    for wav in folder.glob("**/*.wav"):

        speaker = normalize(wav.parent.name)

        try:
            torchaudio.load(wav)
        except:
            invalid_files.append(str(wav))
            continue

        rows.append({
            "file": wav.name,
            "label": label,
            "path": str(wav),
            "speaker": speaker
        })


process_folder(DEMENTIA_DIR, "dementia")
process_folder(NODEMENTIA_DIR, "nodementia")

df = pd.DataFrame(rows)

print("Total valid audio files:", len(df))

Total valid audio files: 455


**Assign Dataset Split**

Assign each audio recording to train, validation, or test based on its speaker to ensure recordings from the same speaker never appear in multiple splits.

In [153]:
def assign_split(speaker):

    if speaker in train_speakers:
        return "train"

    if speaker in valid_speakers:
        return "valid"

    if speaker in test_speakers:
        return "test"

    return None

print("DF columns:", df.columns)
print("Number of rows:", len(df))
print(df.head())

df["split"] = df["speaker"].apply(assign_split)

df = df.dropna(subset=["split"])

DF columns: Index(['file', 'label', 'path', 'speaker'], dtype='object')
Number of rows: 455
                  file     label  \
0        Vampiro_5.wav  dementia   
1        Vampiro_0.wav  dementia   
2   vivnicholson_5.wav  dementia   
3  TrevorPeacock_5.wav  dementia   
4     TonyParkes_5.wav  dementia   

                                                path        speaker  
0  /content/drive/My Drive/NeuralNets Project/dat...        vampiro  
1  /content/drive/My Drive/NeuralNets Project/dat...        vampiro  
2  /content/drive/My Drive/NeuralNets Project/dat...   vivnicholson  
3  /content/drive/My Drive/NeuralNets Project/dat...  trevorpeacock  
4  /content/drive/My Drive/NeuralNets Project/dat...     tonyparkes  


**Create Manifest Tables**

Generate structured dataset manifest files that list each recording along with its label and file path for use during model training.

In [154]:
train_df = df[df["split"] == "train"]
valid_df = df[df["split"] == "valid"]
test_df = df[df["split"] == "test"]

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

train: 198
valid: 51
test: 29


**Verify Class Balance**

Check the distribution of dementia and non-dementia samples in each split to confirm that both classes are represented.

In [155]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALID")
print(valid_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

TRAIN
label
nodementia    103
dementia       95
Name: count, dtype: int64

VALID
label
nodementia    28
dementia      23
Name: count, dtype: int64

TEST
label
nodementia    18
dementia      11
Name: count, dtype: int64


**Verify No Speaker Leakage**

Ensure that no speaker appears in more than one dataset split to prevent the model from learning speaker identity instead of the target task.

In [156]:
train_set = set(train_df["speaker"])
valid_set = set(valid_df["speaker"])
test_set = set(test_df["speaker"])

print("train ∩ valid:", train_set & valid_set)
print("train ∩ test:", train_set & test_set)
print("valid ∩ test:", valid_set & test_set)

train ∩ valid: set()
train ∩ test: set()
valid ∩ test: set()


**Save Final Dataset**

Save the cleaned dataset manifests and related files so the dataset splits can be reused consistently in training and evaluation.

In [157]:
train_df[["file", "label", "path"]].to_csv(
    OUTPUT_DIR / "train_dm.csv",
    sep="\t",
    index=False
)

valid_df[["file", "label", "path"]].to_csv(
    OUTPUT_DIR / "valid_dm.csv",
    sep="\t",
    index=False
)

test_df[["file", "label", "path"]].to_csv(
    OUTPUT_DIR / "test_dm.csv",
    sep="\t",
    index=False
)

pd.DataFrame({"invalid_files": invalid_files}).to_csv(
    OUTPUT_DIR / "invalid_audio_files.csv",
    index=False
)

In [158]:
import pandas as pd

train = pd.read_csv(TRAIN_MANIFEST, sep="\t")
valid = pd.read_csv(VALID_MANIFEST, sep="\t")
test = pd.read_csv(TEST_MANIFEST, sep="\t")

print("train:", len(train))
print("valid:", len(valid))
print("test:", len(test))

train: 198
valid: 51
test: 29


**Look for Duplicates**

Check for duplicate audio file paths within each dataset split to ensure that the same recording does not appear multiple times in the dataset.

In [159]:
print(train["path"].duplicated().sum())
print(valid["path"].duplicated().sum())
print(test["path"].duplicated().sum())

0
0
0
